# On-device 실습 빈칸 모음 · 1~5 통합

모든 문제는 **문제 → 빈칸 코드 → 검증 코드 → 접힌 해설·정답** 순서입니다. PyTorch와 Python 표준 라이브러리만 사용하며 CPU에서 실행합니다. 모델·데이터 다운로드, GPU, transformers, KMeans/EAGLE/DFlash 설치가 필요 없습니다.

첫 준비 셀을 실행하고 원하는 문제의 코드와 테스트를 실행하세요. `????`가 포함된 줄을 완성하세요. 빈칸은 `missing()` 호출과 `# ????`로 표시합니다. 해당 호출을 정답 식이나 문장으로 바꾸세요. 미완성 상태는 `NotImplementedError`로 알려주어 문법 오류 때문에 이전 함수가 남는 문제를 줄였습니다. 작성 중 SyntaxError가 났다면 먼저 수정하고 문제 셀을 다시 실행해야 합니다. 테스트 PASS는 작은 입력에서의 동작 확인이며 전체 모델 품질·속도 또는 숙련을 의미하지 않습니다.

원본 실습 구간 32개를 독립 함수로 추출했습니다. 바깥 학습 루프·다운로드는 제외하고 필요한 문맥과 변수는 인자로 제공합니다. 원본은 수정하지 않았습니다. 5번의 이미 답이 채워진 표시 구간도 연습용으로 다시 비웠습니다. 각 문제의 테스트 입력은 두 종류 이상입니다. 총 소요 시간은 개인별로 다르므로 한 번에 한 단원씩 진행하세요.

In [ ]:
import torch
import torch.nn as nn  # nn.Linear, nn.LayerNorm 등
from collections import namedtuple
from types import SimpleNamespace
torch.set_num_threads(1)
Codebook = namedtuple('Codebook', ['centroids', 'labels'])

def missing():
    raise NotImplementedError('???? 표시의 missing()을 정답 코드로 바꾸세요.')

def pseudo_quantize_tensor(w, n_bit, q_group_size):
    # 테스트용 고정 간격 양자화. 실제 그룹 양자화 구현은 평가하지 않습니다.
    assert n_bit == 3 and q_group_size == 2
    return torch.round(w * 4) / 4

def greedy_step(model, x, cache):
    # 실제 모델/캐시 대신 토큰+1, 호출 횟수+1
    return (x + 1) % 10, cache + 1

class DummyTarget:
    def __call__(self, ids, past_key_values, use_cache):
        assert use_cache
        logits = torch.nn.functional.one_hot((ids+1)%10, 10).float()*3
        return SimpleNamespace(logits=logits, past_key_values=past_key_values+ids.shape[1])

class DummyTree:
    def __call__(self, ids, output_orig, past_key_values, position_ids):
        assert output_orig and torch.equal(position_ids, torch.arange(ids.shape[1]).view(1,-1)+4)
        assert past_key_values == 7
        return {'hidden_states': ids.float()}, torch.nn.functional.one_hot(ids%10,10).float(), ids.float().unsqueeze(-1)

class DummyDraft:
    def compute_logits(self, hidden, output_head):
        return output_head(hidden)

class TinyNet(nn.Module):
    def __init__(self, pair=False):
        super().__init__()
        self.hidden = nn.Linear(4, 5)
        self.head = nn.Linear(5, 3)
        self.pair = pair
    def forward(self, x):
        h = torch.tanh(self.hidden(x))
        y = self.head(h)
        return (y,h) if self.pair else y

def snapshot(x):
    if isinstance(x, torch.Tensor): return x.detach().clone()
    if isinstance(x, nn.Module): return {k:v.detach().clone() for k,v in x.state_dict().items()}
    if isinstance(x, dict): return {k:snapshot(v) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return tuple(snapshot(v) for v in x)
    return x

def same(actual, expected):
    if isinstance(expected, torch.Tensor):
        assert isinstance(actual, torch.Tensor), 'Tensor를 반환해야 합니다.'
        torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    elif isinstance(expected, dict):
        assert actual.keys()==expected.keys()
        for k in expected: same(actual[k],expected[k])
    elif isinstance(expected, tuple):
        assert len(actual)==len(expected)
        for a,e in zip(actual,expected): same(a,e)
    elif isinstance(expected,float):
        assert abs(actual-expected)<1e-6, (actual,expected)
    else: assert actual==expected, (actual,expected)


## 01. 실수 나누기와 반올림

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 28, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(scaled_tensor, rounded_tensor)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
fp_tensor=torch.tensor([[-.73,.26,.52],[.11,-.18,.91]])*(case+1)
scale=.1*(case+1)
```

**원본 힌트**
```python
# Step 1: fp_tensor를 scale 하세요.
# Step 2: 부동 소수점 값을 정수 값으로 rounding 하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_01(fp_tensor, scale):
    scaled_tensor = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    rounded_tensor = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (scaled_tensor, rounded_tensor)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_01():
    expected = ((torch.tensor([[-7.300000190734863, 2.5999999046325684, 5.199999809265137], [1.100000023841858, -1.8000000715255737, 9.100000381469727]], dtype=torch.float32), torch.tensor([[-7.0, 3.0, 5.0], [1.0, -2.0, 9.0]], dtype=torch.float32),), (torch.tensor([[-7.300000190734863, 2.5999999046325684, 5.199999809265137], [1.100000023841858, -1.8000000715255737, 9.100000381469727]], dtype=torch.float32), torch.tensor([[-7.0, 3.0, 5.0], [1.0, -2.0, 9.0]], dtype=torch.float32),),)
    for case in range(2):
        torch.manual_seed(1010+case)
        fp_tensor=torch.tensor([[-.73,.26,.52],[.11,-.18,.91]])*(case+1)
        scale=.1*(case+1)
        actual = solve_01(fp_tensor, scale)
        same(snapshot(actual), expected[case])
    print("PASS · 01 · 2개 입력 검증")
try:
    check_01()
except Exception as error:
    print("FAIL · 01 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

q=round(r/s)+z. 먼저 r/s를 구하고 torch.round를 적용합니다. 예: .26/.1=2.6 → 3. 입력과 출력 shape=(2,3).

```python
@torch.no_grad()
def solve_01(fp_tensor, scale):
    scaled_tensor = fp_tensor/scale
    rounded_tensor = torch.round(scaled_tensor)
    return (scaled_tensor, rounded_tensor)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 02. zero point 이동

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 28, 표시 구간 2. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`shifted_tensor`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
rounded_tensor=torch.tensor([[-3,0,2],[4,1,-1]],dtype=torch.int32)
zero_point=case*3-2
```

**원본 힌트**
```python
# Step 3: rounded_tensor를 zero_point 만큼 shift하여 영점을 조정합니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_02(rounded_tensor, zero_point):
    shifted_tensor = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return shifted_tensor

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_02():
    expected = (torch.tensor([[-5, -2, 0], [2, -1, -3]], dtype=torch.int32), torch.tensor([[-2, 1, 3], [5, 2, 0]], dtype=torch.int32),)
    for case in range(2):
        torch.manual_seed(1020+case)
        rounded_tensor=torch.tensor([[-3,0,2],[4,1,-1]],dtype=torch.int32)
        zero_point=case*3-2
        actual = solve_02(rounded_tensor, zero_point)
        same(snapshot(actual), expected[case])
    print("PASS · 02 · 2개 입력 검증")
try:
    check_02()
except Exception as error:
    print("FAIL · 02 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

정수 코드에 z를 더합니다. 예: 3+10=13. 여기서는 clamp 이전 단계만 연습합니다.

```python
@torch.no_grad()
def solve_02(rounded_tensor, zero_point):
    shifted_tensor = rounded_tensor + zero_point
    return shifted_tensor
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 03. scale과 zero point 계산

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 37, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(scale, zero_point)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
fp_max=1.3+case
fp_min=-.7
quantized_min=-8
quantized_max=7
```

**원본 힌트**
```python
# hint: quantized_max - quantized_min = 2 ** bitwith - 1
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_03(fp_max, fp_min, quantized_max, quantized_min):
    scale = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    zero_point = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (scale, zero_point)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_03():
    expected = ((0.13333333333333333, -3,), (0.2, -4,),)
    for case in range(2):
        torch.manual_seed(1030+case)
        fp_max=1.3+case
        fp_min=-.7
        quantized_min=-8
        quantized_max=7
        actual = solve_03(fp_max, fp_min, quantized_max, quantized_min)
        same(snapshot(actual), expected[case])
    print("PASS · 03 · 2개 입력 검증")
try:
    check_03()
except Exception as error:
    print("FAIL · 03 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

s=(rmax-rmin)/(qmax-qmin), z=round(qmin-rmin/s). 예: 범위[-1,1],[-2,1]이면 s=2/3,z=0. Python 숫자이므로 round 사용. 상수 입력은 제외합니다.

```python
@torch.no_grad()
def solve_03(fp_max, fp_min, quantized_max, quantized_min):
    scale = (fp_max - fp_min) / (quantized_max - quantized_min)
    zero_point = round(quantized_min - fp_min/scale)
    return (scale, zero_point)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 04. 정수 Linear 출력 재스케일

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 59, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`output`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
output=torch.arange(6).float().reshape(2,3)-3
input_scale=.2
weight_scale=torch.tensor([.1,.3,.5]).reshape(1,3)
output_scale=.1*(case+1)
output_zero_point=case-2
```

**원본 힌트**
```python
# Step 2: output 텐서를 scale합니다.
# Step 3: output을 out_zero_point 만큼 이동하여 영점을 조정합니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_04(output, input_scale, weight_scale, output_scale, output_zero_point):
    output = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    output = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return output

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_04():
    expected = (torch.tensor([[-2.5999999046325684, -3.200000047683716, -3.0], [-2.0, -1.399999976158142, 0.0]], dtype=torch.float32), torch.tensor([[-1.2999999523162842, -1.600000023841858, -1.5], [-1.0, -0.699999988079071, 0.0]], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1040+case)
        output=torch.arange(6).float().reshape(2,3)-3
        input_scale=.2
        weight_scale=torch.tensor([.1,.3,.5]).reshape(1,3)
        output_scale=.1*(case+1)
        output_zero_point=case-2
        actual = solve_04(output, input_scale, weight_scale, output_scale, output_zero_point)
        same(snapshot(actual), expected[case])
    print("PASS · 04 · 2개 입력 검증")
try:
    check_04()
except Exception as error:
    print("FAIL · 04 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

제공 output은 이미 정수 누적과 bias 보정이 끝난 값입니다. output × (입력 scale × weight scale / 출력 scale) + 출력 z. 예: 10×(.2×.3/.1)+2=8. 채널별 scale의 broadcast를 확인하세요.

```python
@torch.no_grad()
def solve_04(output, input_scale, weight_scale, output_scale, output_zero_point):
    output = output*(input_scale * weight_scale / output_scale)
    output = output + output_zero_point
    return output
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 05. 정수 Conv2d 출력 재스케일

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 67, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`output`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
output=torch.arange(24).float().reshape(2,3,2,2)-3
input_scale=.2
weight_scale=torch.tensor([.1,.3,.5]).reshape(1,3,1,1)
output_scale=.1*(case+1)
output_zero_point=case-2
```

**원본 힌트**
```python
# hint: 이번 코드 블록은 quantized_linear()와 매우 유사합니다.
# Step 2: output 텐서를 scale합니다.
# Step 3: output을 out_zero_point 만큼 이동하여 영점을 조정합니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_05(output, input_scale, weight_scale, output_scale, output_zero_point):
    output = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    output = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return output

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_05():
    expected = (torch.tensor([[[[-2.5999999046325684, -2.4000000953674316], [-2.200000047683716, -2.0]], [[-1.399999976158142, -0.7999999523162842], [-0.19999992847442627, 0.40000009536743164]], [[3.0, 4.0], [5.0, 6.0]]], [[[-0.19999980926513672, 2.384185791015625e-07], [0.20000028610229492, 0.40000009536743164]], [[5.800000190734863, 6.40000057220459], [7.0, 7.600000381469727]], [[15.0, 16.0], [17.0, 18.0]]]], dtype=torch.float32), torch.tensor([[[[-1.2999999523162842, -1.2000000476837158], [-1.100000023841858, -1.0]], [[-0.699999988079071, -0.3999999761581421], [-0.09999996423721313, 0.20000004768371582]], [[1.5, 2.0], [2.5, 3.0]]], [[[-0.09999990463256836, 1.1920928955078125e-07], [0.10000014305114746, 0.20000004768371582]], [[2.9000000953674316, 3.200000286102295], [3.5, 3.8000001907348633]], [[7.5, 8.0], [8.5, 9.0]]]], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1050+case)
        output=torch.arange(24).float().reshape(2,3,2,2)-3
        input_scale=.2
        weight_scale=torch.tensor([.1,.3,.5]).reshape(1,3,1,1)
        output_scale=.1*(case+1)
        output_zero_point=case-2
        actual = solve_05(output, input_scale, weight_scale, output_scale, output_zero_point)
        same(snapshot(actual), expected[case])
    print("PASS · 05 · 2개 입력 검증")
try:
    check_05()
except Exception as error:
    print("FAIL · 05 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

제공 output은 이미 정수 누적과 bias 보정이 끝난 값입니다. output × (입력 scale × weight scale / 출력 scale) + 출력 z. 예: 10×(.2×.3/.1)+2=8. 채널별 scale의 broadcast를 확인하세요.

```python
@torch.no_grad()
def solve_05(output, input_scale, weight_scale, output_scale, output_zero_point):
    output = output*(input_scale * weight_scale / output_scale)
    output = output + output_zero_point
    return output
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 06. 비트 수와 클러스터 개수

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 89, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`n_clusters`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
bitwidth=case+2
```

**원본 힌트**
```python
# bitwidth에 따라 클러스터 수를 설정하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_06(bitwidth):
    n_clusters = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return n_clusters

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_06():
    expected = (4, 8,)
    for case in range(2):
        torch.manual_seed(1060+case)
        bitwidth=case+2
        actual = solve_06(bitwidth)
        same(snapshot(actual), expected[case])
    print("PASS · 06 · 2개 입력 검증")
try:
    check_06()
except Exception as error:
    print("FAIL · 06 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

b비트의 번호는 2**b개입니다. 예: 2비트는 00,01,10,11 → 4개. KMeans 학습은 하지 않습니다.

```python
@torch.no_grad()
def solve_06(bitwidth):
    n_clusters = 2**bitwidth
    return n_clusters
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 07. 코드북 디코딩

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 89, 표시 구간 2. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`quantized_tensor`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
codebook=Codebook(torch.tensor([.2,1.4,-2.])* (case+1),torch.tensor([2,0,2,1,0,1]))
```

**원본 힌트**
```python
# 추론에서 사용할 quantized tensor를 구하기 위해 코드북을 디코딩하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_07(codebook):
    quantized_tensor = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return quantized_tensor

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_07():
    expected = (torch.tensor([-2.0, 0.20000000298023224, -2.0, 1.399999976158142, 0.20000000298023224, 1.399999976158142], dtype=torch.float32), torch.tensor([-4.0, 0.4000000059604645, -4.0, 2.799999952316284, 0.4000000059604645, 2.799999952316284], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1070+case)
        codebook=Codebook(torch.tensor([.2,1.4,-2.])* (case+1),torch.tensor([2,0,2,1,0,1]))
        actual = solve_07(codebook)
        same(snapshot(actual), expected[case])
    print("PASS · 07 · 2개 입력 검증")
try:
    check_07()
except Exception as error:
    print("FAIL · 07 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

centroids는 대표값, labels는 대표값 번호입니다. 예: [.2,1.4,-2][[2,0]] → [-2,.2]. 결과는 labels와 같은 길이의 float 텐서입니다.

```python
@torch.no_grad()
def solve_07(codebook):
    quantized_tensor = codebook.centroids[codebook.labels]
    return quantized_tensor
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 08. 코드북 중심값 갱신

**원본:** `1. [AVD] Basic Quantization Lab.ipynb` 전체 셀 100, 표시 구간 1. 답안: `1. [AVD] Basic Quantization Lab_answer.ipynb`. 교안: `1. Basic Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`codebook.centroids`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
fp32_tensor=torch.tensor([.1,1.1,2.4,.5,1.7,2.8])*(case+1)
codebook=Codebook(torch.zeros(3),torch.tensor([0,1,2,0,1,2]))
```

**필요한 문맥:** 함수 안에서 다음 준비/반복을 포함하세요.
```python
for k in range(codebook.centroids.numel()):
```

**원본 힌트**
```python
# hint : torch.mean() 함수를 이용하여 평균을 구할 수 있습니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_08(fp32_tensor, codebook):
    for k in range(codebook.centroids.numel()):
        codebook.centroids[k] = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return codebook.centroids

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_08():
    expected = (torch.tensor([0.30000001192092896, 1.4000000953674316, 2.5999999046325684], dtype=torch.float32), torch.tensor([0.6000000238418579, 2.8000001907348633, 5.199999809265137], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1080+case)
        fp32_tensor=torch.tensor([.1,1.1,2.4,.5,1.7,2.8])*(case+1)
        codebook=Codebook(torch.zeros(3),torch.tensor([0,1,2,0,1,2]))
        actual = solve_08(fp32_tensor, codebook)
        same(snapshot(actual), expected[case])
    print("PASS · 08 · 2개 입력 검증")
try:
    check_08()
except Exception as error:
    print("FAIL · 08 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

labels==k는 길이6의 bool 텐서입니다. True 위치를 골라 평균냅니다. 예: k=0 → [.1,.5] → .3. 모든 클러스터는 비어 있지 않습니다.

```python
@torch.no_grad()
def solve_08(fp32_tensor, codebook):
    for k in range(codebook.centroids.numel()):
        codebook.centroids[k] = torch.mean(fp32_tensor[codebook.labels==k])
    return codebook.centroids
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 09. 상위 1% 채널 번호

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 31, 표시 구간 1. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`outlier_mask`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
importance=torch.arange(100*(case+1)).float().roll(13)
```

**원본 힌트**
```python
# Step 1: importance를 기준으로 1%의 중요한 채널을 찾으세요  (hint: use torch.topk())
# hint : torch.topk() 함수를 사용하세요. torch.topk() 함수는 PyTorch에서 텐서의 값 중 상위 k개의 값과 그들의 인덱스를 반환하는 함수입니다. torch.topk()[0]는 값을, torch.topk()[1]은 인덱스를 반환합니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_09(importance):
    outlier_mask = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return outlier_mask

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_09():
    expected = (torch.tensor([12], dtype=torch.int64), torch.tensor([12, 11], dtype=torch.int64),)
    for case in range(2):
        torch.manual_seed(1090+case)
        importance=torch.arange(100*(case+1)).float().roll(13)
        actual = solve_09(importance)
        same(snapshot(actual), expected[case])
    print("PASS · 09 · 2개 입력 검증")
try:
    check_09()
except Exception as error:
    print("FAIL · 09 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

topk(x,k)는 (큰 값, 그 위치)를 반환합니다. [1]은 int64 인덱스입니다. 예: 200채널이면 2개 선택, shape=(2,).

```python
@torch.no_grad()
def solve_09(importance):
    outlier_mask = torch.topk(importance, int(len(importance) * 0.01))[1]
    return outlier_mask
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 10. 확대한 열 다시 축소

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 31, 표시 구간 2. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`m`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
m=nn.Linear(4,2,bias=False)
m.weight.data.copy_(torch.arange(8).view(2,4)+1.)
outlier_mask=torch.tensor([1,3])
scale_factor=case+2.
```

**원본 힌트**
```python
# Step 2: scale_factor를 이용해 중요한 가중치 채널의 값을 다시 축소하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_10(m, outlier_mask, scale_factor):
    m.weight.data[:, outlier_mask] /= missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return m

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_10():
    expected = ({'weight': torch.tensor([[1.0, 1.0, 3.0, 2.0], [5.0, 3.0, 7.0, 4.0]], dtype=torch.float32)}, {'weight': torch.tensor([[1.0, 0.6666666865348816, 3.0, 1.3333333730697632], [5.0, 2.0, 7.0, 2.6666667461395264]], dtype=torch.float32)},)
    for case in range(2):
        torch.manual_seed(1100+case)
        m=nn.Linear(4,2,bias=False)
        m.weight.data.copy_(torch.arange(8).view(2,4)+1.)
        outlier_mask=torch.tensor([1,3])
        scale_factor=case+2.
        actual = solve_10(m, outlier_mask, scale_factor)
        same(snapshot(actual), expected[case])
    print("PASS · 10 · 2개 입력 검증")
try:
    check_10()
except Exception as error:
    print("FAIL · 10 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

모든 행에서 선택한 열만 scale_factor로 나눕니다. W(2,4), indices(2,) → 선택(2,2). 예: 선택된6을3으로 나누면2. 앞선 확대·양자화는 이미 완료된 상태입니다.

```python
@torch.no_grad()
def solve_10(m, outlier_mask, scale_factor):
    m.weight.data[:, outlier_mask] /= scale_factor
    return m
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 11. AWQ 스케일 지수

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 38, 표시 구간 1. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`scales`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
s_x=torch.tensor([1.,4.,9.])
ratio=[.5,.25][case]
```

**원본 힌트**
```python
# Step 2: 공식에 따라 스케일 계산
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_11(s_x, ratio):
    scales = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return scales

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_11():
    expected = (torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32), torch.tensor([1.0, 1.4142135381698608, 1.7320507764816284], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1110+case)
        s_x=torch.tensor([1.,4.,9.])
        ratio=[.5,.25][case]
        actual = solve_11(s_x, ratio)
        same(snapshot(actual), expected[case])
    print("PASS · 11 · 2개 입력 검증")
try:
    check_11()
except Exception as error:
    print("FAIL · 11 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

공식 s=s_x^α, ratio=α. 예: [1,4,9]**.5=[1,2,3]. 뒤의 정규화 이전 값입니다.

```python
@torch.no_grad()
def solve_11(s_x, ratio):
    scales = s_x ** ratio
    return scales
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 12. 확대 → 양자화 → 축소

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 38, 표시 구간 2. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`fc`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
fc=nn.Linear(4,2,bias=False)
fc.weight.data.copy_(torch.tensor([[.13,.27,.42,.59],[-.31,.62,.78,-.91]]))
scales=torch.tensor([1.,2.,3.,4.])*(case+1)
w_bit=3
q_group_size=2
```

**원본 힌트**
```python
# Step 3: scale_factor를 이용해 중요한 가중치 채널의 값을 확대합니다.
# Step 4: scale_factor를 이용해 중요한 가중치 채널의 값을 다시 축소하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_12(fc, scales, w_bit, q_group_size):
    fc.weight.data *= missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    fc.weight.data = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    fc.weight.data /= missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return fc

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_12():
    expected = ({'weight': torch.tensor([[0.25, 0.25, 0.4166666567325592, 0.5625], [-0.25, 0.625, 0.75, -0.9375]], dtype=torch.float32)}, {'weight': torch.tensor([[0.125, 0.25, 0.4166666567325592, 0.59375], [-0.25, 0.625, 0.7916666865348816, -0.90625]], dtype=torch.float32)},)
    for case in range(2):
        torch.manual_seed(1120+case)
        fc=nn.Linear(4,2,bias=False)
        fc.weight.data.copy_(torch.tensor([[.13,.27,.42,.59],[-.31,.62,.78,-.91]]))
        scales=torch.tensor([1.,2.,3.,4.])*(case+1)
        w_bit=3
        q_group_size=2
        actual = solve_12(fc, scales, w_bit, q_group_size)
        same(snapshot(actual), expected[case])
    print("PASS · 12 · 2개 입력 검증")
try:
    check_12()
except Exception as error:
    print("FAIL · 12 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

W*s → Q(W*s) → /s. 예: .13×2=.26 → 더미 양자화 .25 → /2=.125. 더미는 .25 간격이고 실제 그룹 양자화 품질은 검사하지 않습니다.

```python
@torch.no_grad()
def solve_12(fc, scales, w_bit, q_group_size):
    fc.weight.data *= scales
    fc.weight.data = pseudo_quantize_tensor(fc.weight.data, w_bit, q_group_size)
    fc.weight.data /= scales
    return fc
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 13. LayerNorm 스케일 이동

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 54, 표시 구간 1. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`ln`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
ln=nn.LayerNorm(3)
ln.weight.data.copy_(torch.tensor([2.,3.,4.]))
ln.bias.data.copy_(torch.tensor([.3,-.4,.5]))
scale=torch.tensor([1.,2.,4.])*(case+1)
```

**원본 힌트**
```python
# Step 1: layernorm의 weight와 bias를 scale로 나누어주세요.
# div_() 함수는 텐서 자체를 스케일로 나누는 인플레이스 연산
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_13(ln, scale):
    missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return ln

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_13():
    expected = ({'weight': torch.tensor([2.0, 1.5, 1.0], dtype=torch.float32), 'bias': torch.tensor([0.30000001192092896, -0.20000000298023224, 0.125], dtype=torch.float32)}, {'weight': torch.tensor([1.0, 0.75, 0.5], dtype=torch.float32), 'bias': torch.tensor([0.15000000596046448, -0.10000000149011612, 0.0625], dtype=torch.float32)},)
    for case in range(2):
        torch.manual_seed(1130+case)
        ln=nn.LayerNorm(3)
        ln.weight.data.copy_(torch.tensor([2.,3.,4.]))
        ln.bias.data.copy_(torch.tensor([.3,-.4,.5]))
        scale=torch.tensor([1.,2.,4.])*(case+1)
        actual = solve_13(ln, scale)
        same(snapshot(actual), expected[case])
    print("PASS · 13 · 2개 입력 검증")
try:
    check_13()
except Exception as error:
    print("FAIL · 13 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

LN(x)=γ*xhat+β. γ와β 모두 s로 나누면 LN출력/s가 됩니다. 예: (2*xhat+4)/2=xhat+2.

```python
@torch.no_grad()
def solve_13(ln, scale):
    ln.weight.div_(scale)
    ln.bias.div_(scale)
    return ln
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 14. 후속 FC 보상

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 54, 표시 구간 2. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`fc`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
fc=nn.Linear(3,2)
fc.weight.data.copy_(torch.arange(6).reshape(2,3)+1.)
scale=torch.tensor([1.,2.,4.])*(case+1)
```

**원본 힌트**
```python
# Step 2: fc의 weight에 scale을 곱해주세요.
# mul_() 함수는 텐서 자체에 스케일을 곱하는 인플레이스 연산
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_14(fc, scale):
    missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return fc

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_14():
    expected = ({'weight': torch.tensor([[1.0, 4.0, 12.0], [4.0, 10.0, 24.0]], dtype=torch.float32), 'bias': torch.tensor([0.3127572536468506, 0.16506928205490112], dtype=torch.float32)}, {'weight': torch.tensor([[2.0, 8.0, 24.0], [8.0, 20.0, 48.0]], dtype=torch.float32), 'bias': torch.tensor([0.09474587440490723, 0.2995569705963135], dtype=torch.float32)},)
    for case in range(2):
        torch.manual_seed(1140+case)
        fc=nn.Linear(3,2)
        fc.weight.data.copy_(torch.arange(6).reshape(2,3)+1.)
        scale=torch.tensor([1.,2.,4.])*(case+1)
        actual = solve_14(fc, scale)
        same(snapshot(actual), expected[case])
    print("PASS · 14 · 2개 입력 검증")
try:
    check_14()
except Exception as error:
    print("FAIL · 14 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

입력/s를 보상하려면 weight 입력 열마다 s를 곱합니다. 예: (12/3)×(2×3)=24. bias는 바꾸지 않습니다.

```python
@torch.no_grad()
def solve_14(fc, scale):
    fc.weight.mul_(scale)
    return fc
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 15. SmoothQuant 스케일 공식

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 59, 표시 구간 1. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`scales`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
act_scales=torch.tensor([1.,4.,9.])
weight_scales=torch.tensor([4.,1.,9.])
alpha=[.5,.25][case]
```

**원본 힌트**
```python
#Activation Scales 값과 Weight Scales 값에 alpha를 적절히 거듭제곱해주어야 합니다.
#Hint: pow()함수를 통해서 거듭제곱을 사용할 수 있습니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_15(act_scales, weight_scales, alpha):
    scales = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return scales

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_15():
    expected = (torch.tensor([0.5, 2.0, 1.0], dtype=torch.float32), torch.tensor([0.3535533845424652, 1.4142135381698608, 0.3333333432674408], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1150+case)
        act_scales=torch.tensor([1.,4.,9.])
        weight_scales=torch.tensor([4.,1.,9.])
        alpha=[.5,.25][case]
        actual = solve_15(act_scales, weight_scales, alpha)
        same(snapshot(actual), expected[case])
    print("PASS · 15 · 2개 입력 검증")
try:
    check_15()
except Exception as error:
    print("FAIL · 15 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

s=a^α / w^(1-α). α=.5이면 sqrt(a/w). 예: a=4,w=1 → s=2. 모든 입력 통계는 양수입니다.

```python
@torch.no_grad()
def solve_15(act_scales, weight_scales, alpha):
    scales = act_scales.pow(alpha) / weight_scales.pow(1 - alpha)
    return scales
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 16. 회전 행렬 방향

**원본:** `2. [AVD] Advanced Quantization Lab.ipynb` 전체 셀 71, 표시 구간 1. 답안: `2. [AVD] Advanced Quantization Lab_answer.ipynb`. 교안: `2. Advanced Quantization_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`m`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
n=['embed','o_proj','q_proj'][case]
m=nn.Embedding(3,2) if case==0 else nn.Linear(3,2,bias=False) if case==1 else nn.Linear(2,3,bias=False)
m.weight.data.copy_(torch.arange(6).reshape_as(m.weight).float())
R1=torch.tensor([[0.,-1.],[1.,0.]])
```

**원본 힌트**
```python
# Pytorch에서 '@' 연산자는 행렬 곱 (dot product)을 의미합니다.
# nn.Linear의 weight 파라미터는 W^T 형태로 저장되어 있다는 점을 유의해야 합니다.
#
# Shape 요약:
#   Embedding  : (num_tokens, hidden_dim)
#   Linear     : (out_channels, in_channels)
#   Rotation R : (hidden_dim, hidden_dim)
# Embedding Weight: (num_tokens, hidden_dim)
# Linear Layer: (out_features, in_features)
# Attention 출력 또는 FFN 다운프로젝션
# Q, K, V 프로젝션 및 FFN 게이트/업프로젝션
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_16(m, n, R1):
    if isinstance(m, nn.Embedding):
        W_ = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        m.weight.data = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    if isinstance(m, nn.Linear):
        if 'o_proj' in n or 'down_proj' in n:
            W_ = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
            m.weight.data = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        else:
            W_ = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
            m.weight.data = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return m

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_16():
    expected = ({'weight': torch.tensor([[1.0, 0.0], [3.0, -2.0], [5.0, -4.0]], dtype=torch.float32)}, {'weight': torch.tensor([[3.0, 4.0, 5.0], [0.0, -1.0, -2.0]], dtype=torch.float32)}, {'weight': torch.tensor([[1.0, 0.0], [3.0, -2.0], [5.0, -4.0]], dtype=torch.float32)},)
    for case in range(3):
        torch.manual_seed(1160+case)
        n=['embed','o_proj','q_proj'][case]
        m=nn.Embedding(3,2) if case==0 else nn.Linear(3,2,bias=False) if case==1 else nn.Linear(2,3,bias=False)
        m.weight.data.copy_(torch.arange(6).reshape_as(m.weight).float())
        R1=torch.tensor([[0.,-1.],[1.,0.]])
        actual = solve_16(m, n, R1)
        same(snapshot(actual), expected[case])
    print("PASS · 16 · 3개 입력 검증")
try:
    check_16()
except Exception as error:
    print("FAIL · 16 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

Embedding과 일반 Linear는 W@R, o_proj/down_proj는 R.T@W. shape가 맞는 쪽에서 곱합니다. 회전은 단일 레이어의 동일 출력 자체를 보장하는 것이 아니라 모델의 대응 변환 일부입니다.

```python
@torch.no_grad()
def solve_16(m, n, R1):
    if isinstance(m, nn.Embedding):
        W_ = m.weight.data
        m.weight.data = W_ @ R1
    if isinstance(m, nn.Linear):
        if "o_proj" in n or "down_proj" in n:
            W_ = m.weight.data
            m.weight.data = R1.T @ W_
        else:
            W_ = m.weight.data
            m.weight.data = W_ @ R1
    return m
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 17. 지식증류 soft

**원본:** `3. [AVD] Knowledge Distillation.ipynb` 전체 셀 27, 표시 구간 1. 답안: `3. [AVD] Knowledge Distillation_answer.ipynb`. 교안: `3. Knowledge Distillation_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`loss`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
teacher=TinyNet(pair=False)
student=TinyNet(pair=False)
teacher.eval()
student.train()
inputs=torch.randn(case+2,4)
labels=torch.arange(case+2)%3
ce_loss=nn.CrossEntropyLoss()
T=case+2.
soft_target_loss_weight=.3+case*.2
ce_loss_weight=.7-case*.2
cosine_loss=nn.CosineEmbeddingLoss()
mse_loss=nn.MSELoss()
```

**원본 힌트**
```python
# Forward pass with the teacher model - do not save gradients here as we do not change the teacher's weights
# Forward pass with the student model
# Soften the student logits by applying softmax
# Hint: nn.functional.softmax()
# Calculate the soft targets loss. Scaled by T**2 as suggested by the authors of the paper "Distilling the knowledge in a neural network"
# Calculate the true label loss
# Weighted sum of the two losses
```


In [ ]:
import torch
import torch.nn as nn

def solve_17(teacher, student, inputs, labels, ce_loss, T, soft_target_loss_weight, ce_loss_weight):
    with torch.no_grad():
        teacher_logits = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    student_logits = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    soft_targets = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    student_prob = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    soft_targets_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    label_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return loss

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_17():
    expected = (torch.tensor(0.9690182209014893, dtype=torch.float32), torch.tensor(0.5685326457023621, dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1170+case)
        teacher=TinyNet(pair=False)
        student=TinyNet(pair=False)
        teacher.eval()
        student.train()
        inputs=torch.randn(case+2,4)
        labels=torch.arange(case+2)%3
        ce_loss=nn.CrossEntropyLoss()
        T=case+2.
        soft_target_loss_weight=.3+case*.2
        ce_loss_weight=.7-case*.2
        cosine_loss=nn.CosineEmbeddingLoss()
        mse_loss=nn.MSELoss()
        actual = solve_17(teacher, student, inputs, labels, ce_loss, T, soft_target_loss_weight, ce_loss_weight)
        same(snapshot(actual), expected[case])
        actual.backward()
        assert all(p.grad is None for p in teacher.parameters()), "교사 gradient가 생성됐습니다."
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in student.parameters()), "학생 gradient를 확인하세요."
    print("PASS · 17 · 2개 입력 검증")
try:
    check_17()
except Exception as error:
    print("FAIL · 17 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

p=softmax(teacher/T),q=softmax(student/T). soft loss=T²*mean_batch(sum_class p*log(p/q)). 최종=soft가중치×KL+CE가중치×CE. 예: 두 분포가 같으면 KL=0.

```python
def solve_17(teacher, student, inputs, labels, ce_loss, T, soft_target_loss_weight, ce_loss_weight):
    with torch.no_grad():
        teacher_logits = teacher(inputs)
    student_logits = student(inputs)
    soft_targets = nn.functional.softmax(teacher_logits / T, dim = -1)
    student_prob = nn.functional.softmax(student_logits / T, dim = -1)
    soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - student_prob.log()), dim=1).mean() * (T**2)
    label_loss = ce_loss(student_logits, labels)
    loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss
    return loss
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 18. 지식증류 cosine

**원본:** `3. [AVD] Knowledge Distillation.ipynb` 전체 셀 40, 표시 구간 1. 답안: `3. [AVD] Knowledge Distillation_answer.ipynb`. 교안: `3. Knowledge Distillation_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`loss`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
teacher=TinyNet(pair=True)
student=TinyNet(pair=True)
teacher.eval()
student.train()
inputs=torch.randn(case+2,4)
labels=torch.arange(case+2)%3
ce_loss=nn.CrossEntropyLoss()
T=case+2.
hidden_rep_loss_weight=.3+case*.2
ce_loss_weight=.7-case*.2
cosine_loss=nn.CosineEmbeddingLoss()
mse_loss=nn.MSELoss()
```

**원본 힌트**
```python
# Forward pass with the teacher model and keep only the hidden representation
# Forward pass with the student model
# Calculate the cosine loss. Target is a vector of ones. From the loss formula above we can see that is
# the case where loss minimization leads to cosine similarity increase.
# Hint: cosine_loss(x, y, target)에서 target은 1로 이루어진 vector이며, torch.ones(inputs.size(0)).cuda())를 사용
# Calculate the true label loss
# Weighted sum of the two losses
```


In [ ]:
import torch
import torch.nn as nn

def solve_18(teacher, student, inputs, labels, ce_loss, cosine_loss, hidden_rep_loss_weight, ce_loss_weight):
    with torch.no_grad():
        _, teacher_hidden_representation = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    student_logits, student_hidden_representation = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    hidden_rep_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    label_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return loss

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_18():
    expected = (torch.tensor(1.3036251068115234, dtype=torch.float32), torch.tensor(1.0937656164169312, dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1180+case)
        teacher=TinyNet(pair=True)
        student=TinyNet(pair=True)
        teacher.eval()
        student.train()
        inputs=torch.randn(case+2,4)
        labels=torch.arange(case+2)%3
        ce_loss=nn.CrossEntropyLoss()
        T=case+2.
        hidden_rep_loss_weight=.3+case*.2
        ce_loss_weight=.7-case*.2
        cosine_loss=nn.CosineEmbeddingLoss()
        mse_loss=nn.MSELoss()
        actual = solve_18(teacher, student, inputs, labels, ce_loss, cosine_loss, hidden_rep_loss_weight, ce_loss_weight)
        same(snapshot(actual), expected[case])
        actual.backward()
        assert all(p.grad is None for p in teacher.parameters()), "교사 gradient가 생성됐습니다."
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in student.parameters()), "학생 gradient를 확인하세요."
    print("PASS · 18 · 2개 입력 검증")
try:
    check_18()
except Exception as error:
    print("FAIL · 18 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

target=+1이면 hidden 방향이 같아지도록 합니다. 예: [1,0],[2,0]은 cosine loss=0. logits에는 정답 CE를 더합니다. 원본 .cuda()를 .to(inputs.device)로 바꿨습니다.

```python
def solve_18(teacher, student, inputs, labels, ce_loss, cosine_loss, hidden_rep_loss_weight, ce_loss_weight):
    with torch.no_grad():
        _, teacher_hidden_representation = teacher(inputs)
    student_logits, student_hidden_representation = student(inputs)
    hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target=torch.ones(inputs.size(0)).to(inputs.device))
    label_loss = ce_loss(student_logits, labels)
    loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 19. 지식증류 mse

**원본:** `3. [AVD] Knowledge Distillation.ipynb` 전체 셀 53, 표시 구간 1. 답안: `3. [AVD] Knowledge Distillation_answer.ipynb`. 교안: `3. Knowledge Distillation_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`loss`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
teacher=TinyNet(pair=True)
student=TinyNet(pair=True)
teacher.eval()
student.train()
inputs=torch.randn(case+2,4)
labels=torch.arange(case+2)%3
ce_loss=nn.CrossEntropyLoss()
T=case+2.
feature_map_weight=.3+case*.2
ce_loss_weight=.7-case*.2
cosine_loss=nn.CosineEmbeddingLoss()
mse_loss=nn.MSELoss()
```

**원본 힌트**
```python
# Again ignore teacher logits
# Forward pass with the student model
# Calculate the loss
# Calculate the true label loss
# Weighted sum of the two losses
```


In [ ]:
import torch
import torch.nn as nn

def solve_19(teacher, student, inputs, labels, ce_loss, mse_loss, feature_map_weight, ce_loss_weight):
    with torch.no_grad():
        _, teacher_feature_map = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    student_logits, regressor_feature_map = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    hidden_rep_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    label_loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    loss = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return loss

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_19():
    expected = (torch.tensor(1.215442180633545, dtype=torch.float32), torch.tensor(0.8176851868629456, dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1190+case)
        teacher=TinyNet(pair=True)
        student=TinyNet(pair=True)
        teacher.eval()
        student.train()
        inputs=torch.randn(case+2,4)
        labels=torch.arange(case+2)%3
        ce_loss=nn.CrossEntropyLoss()
        T=case+2.
        feature_map_weight=.3+case*.2
        ce_loss_weight=.7-case*.2
        cosine_loss=nn.CosineEmbeddingLoss()
        mse_loss=nn.MSELoss()
        actual = solve_19(teacher, student, inputs, labels, ce_loss, mse_loss, feature_map_weight, ce_loss_weight)
        same(snapshot(actual), expected[case])
        actual.backward()
        assert all(p.grad is None for p in teacher.parameters()), "교사 gradient가 생성됐습니다."
        assert any(p.grad is not None and p.grad.abs().sum()>0 for p in student.parameters()), "학생 gradient를 확인하세요."
    print("PASS · 19 · 2개 입력 검증")
try:
    check_19()
except Exception as error:
    print("FAIL · 19 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

학생 feature와 교사 feature의 원소별 제곱차 평균을 CE와 결합합니다. 예: [1,2]와[1,4]의 MSE=2. 더미 feature shape=(B,5).

```python
def solve_19(teacher, student, inputs, labels, ce_loss, mse_loss, feature_map_weight, ce_loss_weight):
    with torch.no_grad():
        _, teacher_feature_map = teacher(inputs)
    student_logits, regressor_feature_map = student(inputs)
    hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)
    label_loss = ce_loss(student_logits, labels)
    loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss
    return loss
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 20. KV Cache 바이트 계산

**원본:** `4. [AVD] KV Cache Pruning.ipynb` 전체 셀 11, 표시 구간 1. 답안: `4. [AVD] KV Cache Pruning answer.ipynb`. 교안: `4. KV Cache Pruning_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`total_bytes`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
batch_size=case+1
n_layers=3
n_kv_heads=2
seq_len=5+case
head_dim=4
dtype_bytes=2
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_20(batch_size, n_layers, n_kv_heads, seq_len, head_dim, dtype_bytes):
    total_bytes = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return total_bytes

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_20():
    expected = (480, 1152,)
    for case in range(2):
        torch.manual_seed(1200+case)
        batch_size=case+1
        n_layers=3
        n_kv_heads=2
        seq_len=5+case
        head_dim=4
        dtype_bytes=2
        actual = solve_20(batch_size, n_layers, n_kv_heads, seq_len, head_dim, dtype_bytes)
        same(snapshot(actual), expected[case])
    print("PASS · 20 · 2개 입력 검증")
try:
    check_20()
except Exception as error:
    print("FAIL · 20 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

2(K,V)×B×층수×KV헤드×토큰수×헤드차원×원소바이트. 예: B=1,L=3,H=2,S=5,D=4,2bytes → 480bytes. model.config 전역 의존은 제거했습니다.

```python
@torch.no_grad()
def solve_20(batch_size, n_layers, n_kv_heads, seq_len, head_dim, dtype_bytes):
    total_bytes = 2 * batch_size * n_layers * n_kv_heads * seq_len * head_dim * dtype_bytes
    return total_bytes
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 21. H2O 점수

**원본:** `4. [AVD] KV Cache Pruning.ipynb` 전체 셀 22, 표시 구간 1. 답안: `4. [AVD] KV Cache Pruning answer.ipynb`. 교안: `4. KV Cache Pruning_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`scores`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
attn_kv=torch.arange(2*3*5*6).float().reshape(2,3,5,6)/100
recent_size=case+1
```

**원본 힌트**
```python
# 전체 query의 attention 누적
# 최근 recent_size개 token은 항상 보존
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_21(attn_kv, recent_size):
    scores = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    scores[..., -recent_size:] = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return scores

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_21():
    expected = (torch.tensor([[[0.6000000238418579, 0.6499999761581421, 0.699999988079071, 0.75, 0.7999999523162842, float('inf')], [2.1000001430511475, 2.1500000953674316, 2.200000047683716, 2.25, 2.299999952316284, float('inf')], [3.5999999046325684, 3.6500000953674316, 3.6999998092651367, 3.75, 3.799999952316284, float('inf')]], [[5.099999904632568, 5.150000095367432, 5.199999809265137, 5.25, 5.299999713897705, float('inf')], [6.600000381469727, 6.649999618530273, 6.700000286102295, 6.75, 6.800000190734863, float('inf')], [8.09999942779541, 8.149999618530273, 8.199999809265137, 8.25, 8.300000190734863, float('inf')]]], dtype=torch.float32), torch.tensor([[[0.6000000238418579, 0.6499999761581421, 0.699999988079071, 0.75, float('inf'), float('inf')], [2.1000001430511475, 2.1500000953674316, 2.200000047683716, 2.25, float('inf'), float('inf')], [3.5999999046325684, 3.6500000953674316, 3.6999998092651367, 3.75, float('inf'), float('inf')]], [[5.099999904632568, 5.150000095367432, 5.199999809265137, 5.25, float('inf'), float('inf')], [6.600000381469727, 6.649999618530273, 6.700000286102295, 6.75, float('inf'), float('inf')], [8.09999942779541, 8.149999618530273, 8.199999809265137, 8.25, float('inf'), float('inf')]]], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1210+case)
        attn_kv=torch.arange(2*3*5*6).float().reshape(2,3,5,6)/100
        recent_size=case+1
        actual = solve_21(attn_kv, recent_size)
        same(snapshot(actual), expected[case])
    print("PASS · 21 · 2개 입력 검증")
try:
    check_21()
except Exception as error:
    print("FAIL · 21 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

Query축(-2)을 합해 (B,H,K)로 만듭니다. 최근 recent_size개의 Key에 inf를 넣습니다. 예: (2,3,5,6) → (2,3,6). recent_size>0인 범위입니다.

```python
@torch.no_grad()
def solve_21(attn_kv, recent_size):
    scores = attn_kv.sum(dim=-2)
    scores[..., -recent_size:] = float("inf")
    return scores
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 22. SnapKV 관찰창과 주변 최대

**원본:** `4. [AVD] KV Cache Pruning.ipynb` 전체 셀 28, 표시 구간 1. 답안: `4. [AVD] KV Cache Pruning answer.ipynb`. 교안: `4. KV Cache Pruning_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`scores`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
attn_kv=torch.rand(2,3,6,8)
w=case+1
K=8
kernel=[3,5][case]
```

**원본 힌트**
```python
# TODO 1
# 마지막 w개의 query가 각 KV token에 준 attention을 합산하세요.
# Hint: attn_kv shape = [B, H, Q, K]
# 각 token 주변 kernel 범위에서 가장 큰 score를 가져옵니다.
# TODO 2
# Hint: start ~ end 위치 중 최대 score
# Observation window는 항상 유지
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_22(attn_kv, w, K, kernel):
    scores = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    half = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    pooled_scores = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    for j in range(K):
        start = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        end = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        pooled_scores[:, :, j] = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    scores = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    scores[:, :, -w:] = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return scores

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_22():
    expected = (torch.tensor([[[0.36655867099761963, 0.9902102947235107, 0.9902102947235107, 0.9902102947235107, 0.9268977046012878, 0.9268977046012878, 0.24873435497283936, float('inf')], [0.5433693528175354, 0.6513189673423767, 0.9714928269386292, 0.9714928269386292, 0.9714928269386292, 0.7498315572738647, 0.7498315572738647, float('inf')], [0.6944734454154968, 0.6944734454154968, 0.6944734454154968, 0.32023435831069946, 0.32023435831069946, 0.32023435831069946, 0.6112785339355469, float('inf')]], [[0.8749484419822693, 0.8749484419822693, 0.41897064447402954, 0.6832737326622009, 0.864304780960083, 0.864304780960083, 0.864304780960083, float('inf')], [0.7626241445541382, 0.7626241445541382, 0.7626241445541382, 0.4186975955963135, 0.9681755304336548, 0.9681755304336548, 0.9681755304336548, float('inf')], [0.48917895555496216, 0.48917895555496216, 0.48917895555496216, 0.29591935873031616, 0.40072548389434814, 0.40072548389434814, 0.46335315704345703, float('inf')]]], dtype=torch.float32), torch.tensor([[[1.3030683994293213, 1.4797426462173462, 1.4797426462173462, 1.5013751983642578, 1.6226974725723267, 1.6226974725723267, float('inf'), float('inf')], [0.9944878220558167, 0.9944878220558167, 1.8067264556884766, 1.8067264556884766, 1.8067264556884766, 1.8067264556884766, float('inf'), float('inf')], [1.1243846416473389, 1.1243846416473389, 1.1243846416473389, 1.4420065879821777, 1.6221227645874023, 1.6221227645874023, float('inf'), float('inf')]], [[1.2543097734451294, 1.2543097734451294, 1.5854980945587158, 1.5854980945587158, 1.5854980945587158, 1.5854980945587158, float('inf'), float('inf')], [1.1112816333770752, 1.1112816333770752, 1.1112816333770752, 1.1112816333770752, 1.839277982711792, 1.839277982711792, float('inf'), float('inf')], [1.757981300354004, 1.757981300354004, 1.757981300354004, 1.757981300354004, 1.757981300354004, 1.1282052993774414, float('inf'), float('inf')]]], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1220+case)
        attn_kv=torch.rand(2,3,6,8)
        w=case+1
        K=8
        kernel=[3,5][case]
        actual = solve_22(attn_kv, w, K, kernel)
        same(snapshot(actual), expected[case])
    print("PASS · 22 · 2개 입력 검증")
try:
    check_22()
except Exception as error:
    print("FAIL · 22 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

최근 w개 Query 합산 → Key축 이웃 최대값 → 최근 w개 Key inf. 예: [.2,1,.3,.3,.2],kernel3 → [1,1,1,.3,.3]. kernel은 홀수, w는 양수입니다.

```python
@torch.no_grad()
def solve_22(attn_kv, w, K, kernel):
    scores = attn_kv[:, :, -w:, :].sum(dim=2)
    half = kernel // 2
    pooled_scores = scores.clone()
    for j in range(K):
        start = max(0, j - half)
        end   = min(K, j + half + 1)
        pooled_scores[:, :, j] = scores[:, :, start:end].max(dim=-1).values
    scores = pooled_scores
    scores[:, :, -w:] = float("inf")
    return scores
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 23. Magnitude 가지치기 마스크

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 10, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`mask`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
W=torch.tensor([[.1,-.8,.3,.6],[-.2,.9,-.4,.7]])
sparsity=[.25,.5][case]
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_23(W, sparsity):
    num_elements = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    num_zeros = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    importance = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    threshold = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    mask = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return mask

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_23():
    expected = (torch.tensor([[False, True, True, True], [False, True, True, True]], dtype=torch.bool), torch.tensor([[False, True, False, True], [False, True, False, True]], dtype=torch.bool),)
    for case in range(2):
        torch.manual_seed(1230+case)
        W=torch.tensor([[.1,-.8,.3,.6],[-.2,.9,-.4,.7]])
        sparsity=[.25,.5][case]
        actual = solve_23(W, sparsity)
        same(snapshot(actual), expected[case])
    print("PASS · 23 · 2개 입력 검증")
try:
    check_23()
except Exception as error:
    print("FAIL · 23 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

abs(W)에서 k번째 작은 값을 기준으로 > 마스크 생성. 예: [.1,.8,.3,.2],50% → [F,T,T,F]. 동점 없는 입력, k>=1입니다. 반환 mask를 W에 곱하는 단계는 원본 제공 코드입니다.

```python
@torch.no_grad()
def solve_23(W, sparsity):
    num_elements = W.numel()
    num_zeros = round(num_elements * sparsity)
    importance = torch.abs(W)
    threshold = torch.kthvalue(importance.flatten(), num_zeros)[0]
    mask = importance > threshold
    return mask
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 24. Wanda 활성값 L2 제곱

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 16, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`activation_norm`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
x=torch.tensor([[1.,2.,3.],[2.,0.,1.]])*(case+1)
```

**원본 힌트**
```python
# activation_norm을 계산하세요.
# x.shape => (hidden_size, batch_size)
# activation_norm.shape => (hidden_size)
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_24(x):
    activation_norm = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return activation_norm

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_24():
    expected = (torch.tensor([14.000000953674316, 5.0], dtype=torch.float32), torch.tensor([56.000003814697266, 20.0], dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(1240+case)
        x=torch.tensor([[1.,2.,3.],[2.,0.,1.]])*(case+1)
        actual = solve_24(x)
        same(snapshot(actual), expected[case])
    print("PASS · 24 · 2개 입력 검증")
try:
    check_24()
except Exception as error:
    print("FAIL · 24 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

x=(입력채널,샘플), 행별 sum(x²). 예: [1,2,3] → 1+4+9=14. 전체 합이나 평균과 다릅니다.

```python
@torch.no_grad()
def solve_24(x):
    activation_norm = torch.norm(x, p=2, dim=1) ** 2
    return activation_norm
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 25. Wanda 행별 마스크

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 19, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`mask`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
W=torch.tensor([[.1,-.8,.3,.4],[.2,.5,-.6,.1]])
sparsity=[.25,.5][case]
n='linear'
input_feat={n:torch.tensor([10.,1.,2.,1.])}
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_25(W, sparsity, input_feat, n):
    row, col = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    num_zeros_per_row = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    importance = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    threshold = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    mask = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return mask

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_25():
    expected = (torch.tensor([[True, True, True, False], [True, True, True, False]], dtype=torch.bool), torch.tensor([[True, True, False, False], [True, False, True, False]], dtype=torch.bool),)
    for case in range(2):
        torch.manual_seed(1250+case)
        W=torch.tensor([[.1,-.8,.3,.4],[.2,.5,-.6,.1]])
        sparsity=[.25,.5][case]
        n='linear'
        input_feat={n:torch.tensor([10.,1.,2.,1.])}
        actual = solve_25(W, sparsity, input_feat, n)
        same(snapshot(actual), expected[case])
    print("PASS · 25 · 2개 입력 검증")
try:
    check_25()
except Exception as error:
    print("FAIL · 25 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

importance=abs(W)×입력채널 크기. 각 행에서 k번째 작은 값을 구해 (row,1)로 세워 비교합니다. 예: 첫 행 중요도=[1,.8,.6,.4],50% → [T,T,F,F].

```python
@torch.no_grad()
def solve_25(W, sparsity, input_feat, n):
    row, col = W.shape
    num_zeros_per_row = round(col * sparsity)
    importance = torch.abs(W) * input_feat[n]
    threshold = torch.kthvalue(importance, num_zeros_per_row, dim=1)[0]
    mask = importance > threshold.reshape(row, 1)
    return mask
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 26. 순차 Draft 생성

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 28, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(torch.cat(tokens,dim=-1), cache)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
model=None
x=torch.tensor([[8-case]])
cache=2
gamma=case+2
```

**필요한 문맥:** 함수 안에서 다음 준비/반복을 포함하세요.
```python
tokens = []
```

**원본 힌트**
```python
# TODO: 다음 draft token을 생성하세요.
# HINT: greedy_step()을 사용해 draft token을 gamma개 순차적으로 생성하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_26(model, x, cache, gamma):
    tokens = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    for _ in range(gamma):
        x, cache = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (torch.cat(tokens,dim=-1), cache)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_26():
    expected = ((torch.tensor([[9, 0]], dtype=torch.int64), 4,), (torch.tensor([[8, 9, 0]], dtype=torch.int64), 5,),)
    for case in range(2):
        torch.manual_seed(1260+case)
        model=None
        x=torch.tensor([[8-case]])
        cache=2
        gamma=case+2
        actual = solve_26(model, x, cache, gamma)
        same(snapshot(actual), expected[case])
    print("PASS · 26 · 2개 입력 검증")
try:
    check_26()
except Exception as error:
    print("FAIL · 26 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

gamma번 greedy_step 호출. 예: 더미 시작8이면 9→0→1. 이전에 생성한 토큰과 갱신한 캐시를 다음 호출에 전달합니다.

```python
@torch.no_grad()
def solve_26(model, x, cache, gamma):
    tokens = []
    for _ in range(gamma):
      x, cache = greedy_step(model, x, cache)
      tokens.append(x)
    return (torch.cat(tokens,dim=-1), cache)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 27. Target에 검증 시퀀스 입력

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 28, 표시 구간 2. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(preds, result.past_key_values)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
target=DummyTarget()
x=torch.tensor([[2+case]])
drafted=torch.tensor([[4,5,6]])
cache=7
```

**원본 힌트**
```python
# TODO : 마지막 committed token과 draft token들을 하나의 sequence로 연결하세요.
# HINT: torch.cat()을 사용해 마지막 committed token과 draft token들을 sequence 방향으로 연결하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_27(target, x, drafted, cache):
    verify_ids = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    result = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    preds = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (preds, result.past_key_values)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_27():
    expected = ((torch.tensor([3, 5, 6, 7], dtype=torch.int64), 11,), (torch.tensor([4, 5, 6, 7], dtype=torch.int64), 11,),)
    for case in range(2):
        torch.manual_seed(1270+case)
        target=DummyTarget()
        x=torch.tensor([[2+case]])
        drafted=torch.tensor([[4,5,6]])
        cache=7
        actual = solve_27(target, x, drafted, cache)
        same(snapshot(actual), expected[case])
    print("PASS · 27 · 2개 입력 검증")
try:
    check_27()
except Exception as error:
    print("FAIL · 27 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

마지막 committed token과 drafted를 길이축으로 cat합니다. 예: [2]+[4,5,6] → [2,4,5,6]. 더미 Target은 각 입력+1을 예측합니다.

```python
@torch.no_grad()
def solve_27(target, x, drafted, cache):
    verify_ids = torch.cat([x, drafted], dim=-1)
    result = target(
      verify_ids,
      past_key_values=cache,
      use_cache=True
    )
    preds = result.logits[0].argmax(-1)
    return (preds, result.past_key_values)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 28. 첫 불일치에서 수락 중단

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 29, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(new_tokens, n)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
drafted=torch.tensor([[1,2,3]])
preds=[torch.tensor([1,9,3,4]),torch.tensor([1,2,3,4]),torch.tensor([9,2,3,4])][case]
gamma=3
```

**원본 힌트**
```python
#TODO: Draft와 Target prediction을 앞에서부터 비교하고 첫 mismatch에서 멈추세요.
# HINT: accept된 draft token 뒤에 Target의 prediction 하나를 추가하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_28(drafted, preds, gamma):
    n = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    for i in range(gamma):
        if drafted[0, i] == preds[i]:
            n += missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
        else:
            break
    new_tokens = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (new_tokens, n)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_28():
    expected = ((torch.tensor([[1, 9]], dtype=torch.int64), 1,), (torch.tensor([[1, 2, 3, 4]], dtype=torch.int64), 3,), (torch.tensor([[9]], dtype=torch.int64), 0,),)
    for case in range(3):
        torch.manual_seed(1280+case)
        drafted=torch.tensor([[1,2,3]])
        preds=[torch.tensor([1,9,3,4]),torch.tensor([1,2,3,4]),torch.tensor([9,2,3,4])][case]
        gamma=3
        actual = solve_28(drafted, preds, gamma)
        same(snapshot(actual), expected[case])
    print("PASS · 28 · 3개 입력 검증")
try:
    check_28()
except Exception as error:
    print("FAIL · 28 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

앞에서 연속 일치한 토큰만 수락하고 Target 토큰 하나 추가. 예: draft[1,2,3],pred[1,9,3,4] → [1,9],n=1. 뒤의 우연한 일치는 수락하지 않습니다.

```python
@torch.no_grad()
def solve_28(drafted, preds, gamma):
    n = 0
    for i in range(gamma):
      if drafted[0, i] == preds[i]:
        n += 1
      else:
        break
    new_tokens = torch.cat([drafted[:, :n],preds[n].view(1, 1)],dim=1)
    return (new_tokens, n)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 29. Tree 조상 마스크 상속

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 40, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`tree_mask`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
mask_index_list=[[0,0,1,3],[0,1,0,2]][case]
tree_mask=torch.eye(5,dtype=torch.bool)
tree_mask[:,0]=True
```

**원본 힌트**
```python
# ToDo: child가 parent가 볼 수 있는 node들을 그대로 볼 수 있게 하세요.
# |= 는 OR 결과를 왼쪽에 저장하는 연산입니다.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_29(tree_mask, mask_index_list):
    for child, parent in enumerate(mask_index_list, start=1):
        tree_mask[child] |= missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return tree_mask

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_29():
    expected = (torch.tensor([[True, False, False, False, False], [True, True, False, False, False], [True, False, True, False, False], [True, True, False, True, False], [True, True, False, True, True]], dtype=torch.bool), torch.tensor([[True, False, False, False, False], [True, True, False, False, False], [True, True, True, False, False], [True, False, False, True, False], [True, True, True, False, True]], dtype=torch.bool),)
    for case in range(2):
        torch.manual_seed(1290+case)
        mask_index_list=[[0,0,1,3],[0,1,0,2]][case]
        tree_mask=torch.eye(5,dtype=torch.bool)
        tree_mask[:,0]=True
        actual = solve_29(tree_mask, mask_index_list)
        same(snapshot(actual), expected[case])
    print("PASS · 29 · 2개 입력 검증")
try:
    check_29()
except Exception as error:
    print("FAIL · 29 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

child가 parent의 가시 범위를 OR로 물려받습니다. 예: 0→1→3이면3은0,1,3을 봅니다. 부모가 자식보다 먼저 나오는 순서이며 bool 마스크입니다.

```python
@torch.no_grad()
def solve_29(tree_mask, mask_index_list):
    for child, parent in enumerate(mask_index_list, start=1):
      tree_mask[child] |= tree_mask[parent]
    return tree_mask
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 30. Tree 전체 한 번에 검증

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 42, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(outputs, tree_logits, hidden_state)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
model=DummyTree()
tree_candidates=torch.tensor([[1,2,3+case]])
past_key_values=7
position_ids=torch.arange(3).view(1,-1)+4
```

**원본 힌트**
```python
# To Do: Tree candidate 전체를 Target model에 한 번에 입력하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_30(model, tree_candidates, past_key_values, position_ids):
    outputs, tree_logits, hidden_state = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (outputs, tree_logits, hidden_state)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_30():
    expected = (({'hidden_states': torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)}, torch.tensor([[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], dtype=torch.float32), torch.tensor([[[1.0], [2.0], [3.0]]], dtype=torch.float32),), ({'hidden_states': torch.tensor([[1.0, 2.0, 4.0]], dtype=torch.float32)}, torch.tensor([[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]]], dtype=torch.float32), torch.tensor([[[1.0], [2.0], [4.0]]], dtype=torch.float32),),)
    for case in range(2):
        torch.manual_seed(1300+case)
        model=DummyTree()
        tree_candidates=torch.tensor([[1,2,3+case]])
        past_key_values=7
        position_ids=torch.arange(3).view(1,-1)+4
        actual = solve_30(model, tree_candidates, past_key_values, position_ids)
        same(snapshot(actual), expected[case])
    print("PASS · 30 · 2개 입력 검증")
try:
    check_30()
except Exception as error:
    print("FAIL · 30 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

후보 전체와 캐시·position_ids를 모델에 전달하고 세 출력을 받습니다. 더미가 전달 인자도 검사합니다. EAGLE 설치는 필요 없습니다.

```python
@torch.no_grad()
def solve_30(model, tree_candidates, past_key_values, position_ids):
    outputs, tree_logits, hidden_state = model(
      tree_candidates,
      output_orig=True,
      past_key_values=past_key_values,
      position_ids=position_ids,
    )
    return (outputs, tree_logits, hidden_state)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 31. 가장 긴 수락 가지 선택

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 43, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`(best_candidate, accept_length)`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
accept_lengths=[torch.tensor([1,3,2]),torch.tensor([2,1,2])][case]
```

**원본 힌트**
```python
# To Do: 가장 길게 일치한 branch와 그 accept length를 선택하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_31(accept_lengths):
    best_candidate = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    accept_length = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return (best_candidate, accept_length)

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_31():
    expected = ((torch.tensor(1, dtype=torch.int64), torch.tensor(3, dtype=torch.int64),), (torch.tensor(0, dtype=torch.int64), torch.tensor(2, dtype=torch.int64),),)
    for case in range(2):
        torch.manual_seed(1310+case)
        accept_lengths=[torch.tensor([1,3,2]),torch.tensor([2,1,2])][case]
        actual = solve_31(accept_lengths)
        same(snapshot(actual), expected[case])
    print("PASS · 31 · 2개 입력 검증")
try:
    check_31()
except Exception as error:
    print("FAIL · 31 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

argmax는 최대값 위치를 반환합니다. 예: [1,3,2] → 가지1,길이3. 동점이면 첫 위치를 선택합니다.

```python
@torch.no_grad()
def solve_31(accept_lengths):
    best_candidate = accept_lengths.argmax()
    accept_length = accept_lengths[best_candidate]
    return (best_candidate, accept_length)
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>

## 32. 병렬 Draft 토큰 선택

**원본:** `5. [AVD] Speculative Decoding.ipynb` 전체 셀 51, 표시 구간 1. 답안: `5. [AVD] Speculative Decoding Answer.ipynb`. 교안: `5. LLM Pruning and Speculative Decoding_slide.pdf`.

다음 원본 구간의 연산을 구현하세요. 인자 이름은 원본과 같습니다. 출력은 **`draft_tokens`** 입니다.

**제공 입력과 shape 확인용 예시** (`case=0`일 때):
```python
case = 0
draft_hidden=torch.arange(24).float().reshape(1,6,4)/10
num_draft=case+2
model=DummyDraft()
output_head=nn.Linear(4,7)
```

**원본 힌트**
```python
# To Do: 마지막 num_draft개의 hidden state만 선택하세요.
# To Do: model.compute_logits()을 사용해 모든 위치의 vocabulary logits을 계산하세요.
# To Do: torch.argmax()를 사용해 모든 위치의 token을 동시에 선택하세요.
```


In [ ]:
import torch
import torch.nn as nn

@torch.no_grad()
def solve_32(draft_hidden, num_draft, model, output_head):
    draft_hidden = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    draft_logits = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    draft_tokens = missing()  # ???? 이 호출을 필요한 식/코드로 바꾸세요.
    return draft_tokens

In [ ]:
# 작은 CPU 입력으로 검증합니다. 이 셀은 수정하지 마세요.
def check_32():
    expected = (torch.tensor([[1, 1]], dtype=torch.int64), torch.tensor([[2, 2, 2]], dtype=torch.int64),)
    for case in range(2):
        torch.manual_seed(1320+case)
        draft_hidden=torch.arange(24).float().reshape(1,6,4)/10
        num_draft=case+2
        model=DummyDraft()
        output_head=nn.Linear(4,7)
        actual = solve_32(draft_hidden, num_draft, model, output_head)
        same(snapshot(actual), expected[case])
    print("PASS · 32 · 2개 입력 검증")
try:
    check_32()
except Exception as error:
    print("FAIL · 32 ·", type(error).__name__, str(error))


<details><summary>해설 · 쉬운 공식 · 정답 보기 (먼저 풀어보세요)</summary>

마지막 num_draft개 hidden을 선택하고 vocabulary logits의 마지막 축 argmax. 예: (1,6,4) → (1,2,4) → (1,2,7) → 토큰(1,2).

```python
@torch.no_grad()
def solve_32(draft_hidden, num_draft, model, output_head):
    draft_hidden = draft_hidden[:, -num_draft:, :]
    draft_logits = model.compute_logits(draft_hidden, output_head)
    draft_tokens = torch.argmax(draft_logits, dim=-1)
    return draft_tokens
```

원본 표시 구간을 함수로 분리했습니다. 반환문은 테스트용으로 추가했습니다.
</details>